# 9. Custo computacional e estabilidade (estudo de caso 1, SECOM)

Este caderno apresenta, em tabelas, os indicadores de **custo computacional** e de **estabilidade de seleção** do estudo de caso 1. Ele **não recalcula nada**: lê as abas já geradas pelo caderno 8 (`8_Analises_Tese.xlsx`) e as organiza como no artigo.

Abas utilizadas:

| Aba | Conteúdo |
|---|---|
| `5_Qualidade_e_Tempo_Selec` | Número de atributos selecionados (`N_*`) por paradigma e ponto de corte |
| `5_Tempo_Curva` | Tempo médio, eficiência (s/atributo) e escalabilidade (expoente alfa de T = c·N^alfa) |
| `11_Estabilidade` | Índice de Nogueira (Phi), intervalo de confiança e classificação, por método |
| `11_Estabilidade_Resumo` | Proporção de métodos estáveis (Phi >= 0,40) por grupo e paradigma |

Sufixos das colunas: `prop` = método proposto, `lit` = literatura, `desc` = estatísticas descritivas; `p_lit` e `p_desc` = valor-p do teste de Mann-Whitney contra a proposta.

**Pré-requisito**: ter executado o caderno 8.

### 1. Configuração

In [ ]:
import os
import pandas as pd

ARQUIVO = "8_Analises_Tese.xlsx"   # saída do caderno 8
PASTA_SAIDA = "9_Custo_e_Estabilidade"
ALFA = 0.05                        # nível de significância do Mann-Whitney
PHI_ESTAVEL = 0.40                 # limiar de estabilidade (Nogueira)

assert os.path.exists(ARQUIVO), f"Arquivo não encontrado: {ARQUIVO}. Execute antes o caderno 8."
os.makedirs(PASTA_SAIDA, exist_ok=True)

def ler(aba):
    return pd.read_excel(ARQUIVO, sheet_name=aba)

pd.options.display.float_format = "{:.3f}".format
pd.options.display.max_columns = 50

### 2. Custo computacional: tempo, eficiência e escalabilidade

In [ ]:
tempo = ler("5_Tempo_Curva")
tempo["Significativo"] = (tempo["p_lit"] <= ALFA) | (tempo["p_desc"] <= ALFA)

colunas = ["P", "Indicador", "prop", "lit", "desc", "p_lit", "p_desc", "Significativo"]
tabela_custo = tempo[colunas].rename(columns={"P": "Paradigma"})
tabela_custo.to_excel(f"{PASTA_SAIDA}/Tabela_Custo.xlsx", index=False)
tabela_custo

### 3. Número de atributos selecionados

Para cada paradigma e ponto de corte (Kneedle e 1 desvio-padrão), mostra o número médio de atributos retidos por grupo de métodos.

In [ ]:
sel = ler("5_Qualidade_e_Tempo_Selec")
colunas = ["P", "T", "N_prop", "N_lit", "N_desc", "N_p_lit", "N_p_desc"]
tabela_n = sel[colunas].rename(columns={"P": "Paradigma", "T": "Ponto de corte"})
tabela_n.to_excel(f"{PASTA_SAIDA}/Tabela_Atributos_Selecionados.xlsx", index=False)
tabela_n

### 4. Estabilidade da seleção (índice de Nogueira)

Phi mede a consistência do conjunto de atributos selecionados entre 100 subamostras estratificadas de 50% da base. Valores a partir de 0,40 indicam seleção estável. As linhas com `Trivial_N_ge_468` verdadeiro correspondem a seleções que retêm praticamente todos os atributos, em que a estabilidade é trivial.

In [ ]:
est = ler("11_Estabilidade")
est["Estavel"] = est["Phi"] >= PHI_ESTAVEL

por_grupo = (est.groupby(["P", "G"])
                .agg(Metodos=("Metodo", "count"),
                     Estaveis=("Estavel", "sum"),
                     Phi_medio=("Phi", "mean"))
                .reset_index()
                .rename(columns={"P": "Paradigma", "G": "Grupo"}))
por_grupo["Pct_estaveis"] = 100 * por_grupo["Estaveis"] / por_grupo["Metodos"]
por_grupo.to_excel(f"{PASTA_SAIDA}/Tabela_Estabilidade_Por_Grupo.xlsx", index=False)
por_grupo

### 5. Resumo da estabilidade por paradigma (proposta vs. literatura)

Tabela já consolidada pelo caderno 8, com a proporção de métodos estáveis em cada grupo e o valor-p da comparação entre proposta e literatura.

In [ ]:
resumo = ler("11_Estabilidade_Resumo").rename(columns={"P": "Paradigma"})
resumo.to_excel(f"{PASTA_SAIDA}/Tabela_Estabilidade_Resumo.xlsx", index=False)
resumo

### 6. Arquivos gerados

Todas as tabelas acima são gravadas em `9_Custo_e_Estabilidade/` (formato Excel).

In [ ]:
print(*sorted(os.listdir(PASTA_SAIDA)), sep="\n")